In [1]:
import pandas as pd
import torch
from tqdm import tqdm
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
from sklearn.metrics import f1_score, cohen_kappa_score, recall_score, accuracy_score, classification_report
from scipy.stats import spearmanr
from torch.utils.data import WeightedRandomSampler


# The following code will only execute
# successfully when compression is complete

import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohabenloughmari/miccai-task2-full")

print("Path to dataset files:", path)

import warnings
warnings.filterwarnings("ignore")

base_path = path
path = os.path.join(base_path, 'Task_2')

df_train = pd.read_csv(os.path.join(path, "df_task2_train.csv"))
df_val = pd.read_csv(os.path.join(path, "df_task2_val.csv"))
df_test = pd.read_csv(os.path.join(path, "df_task2_test.csv"))

device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')


Path to dataset files: /root/.cache/kagglehub/datasets/mohabenloughmari/miccai-task2-full/versions/1


In [2]:

labels = df_train['label'].values.astype(int)
class_counts = np.bincount(labels)
sample_weights = 1.0 / class_counts[labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

n_labels = df_train['label'].nunique()

tabular_cols = ['case', 'label', 'LOCALIZER', 'split_type', 'image']
tab_train = df_train.drop(columns=tabular_cols)
tab_val = df_val.drop(columns=tabular_cols)
tab_test = df_test.drop(columns=tabular_cols)

cat_cols = tab_train.select_dtypes(include='object').columns.tolist()
num_cols = tab_train.select_dtypes(exclude='object').columns.tolist()

print(f"Categorical columns: {cat_cols}")
print(f"Numerical columns: {num_cols}")

tab_train_encoded = pd.get_dummies(tab_train, columns=cat_cols, dtype=np.float32)
tab_val_encoded = pd.get_dummies(tab_val, columns=cat_cols, dtype=np.float32)
tab_test_encoded = pd.get_dummies(tab_test, columns=cat_cols, dtype=np.float32)

tab_val_encoded = tab_val_encoded.reindex(columns=tab_train_encoded.columns, fill_value=0)
tab_test_encoded = tab_test_encoded.reindex(columns=tab_train_encoded.columns, fill_value=0)



Categorical columns: ['side_eye', 'sex']
Numerical columns: ['BScan', 'age', 'num_current_visit']


In [3]:

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [4]:

class CustomImageDataset(Dataset):
    def __init__(self, dataframe, root_dir, tab_data, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.tab_data = torch.tensor(tab_data.values, dtype=torch.float32)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['image'])
        localiser_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['LOCALIZER'])

        image = Image.open(img_name).convert('RGB')
        localiser = Image.open(localiser_name).convert('RGB')

        if self.transform:
            image = self.transform(image)
            localiser = self.transform(localiser)

        label = self.dataframe.iloc[idx]['label']
        tab = self.tab_data[idx]
        return image, localiser, tab, label

In [5]:
train_ds = CustomImageDataset(df_train, os.path.join(path, 'train'), tab_train_encoded, transform=train_transform)  
val_ds = CustomImageDataset(df_val, os.path.join(path, 'val'), tab_val_encoded, transform=test_transform)  
test_ds = CustomImageDataset(df_test, os.path.join(path, 'test'), tab_test_encoded, transform=test_transform)  
  
labels = df_train['label'].values.astype(int)  
class_counts = np.bincount(labels)  
sample_weights = 1.0 / class_counts[labels]  
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))  
  
train_loader = DataLoader(train_ds, batch_size=12, sampler=sampler)  
val_loader = DataLoader(val_ds, batch_size=12, shuffle=False)  
test_loader = DataLoader(test_ds, batch_size=12, shuffle=False)

In [6]:
!pip install open_clip_torch

In [7]:
import torch
import torch.nn as nn
from open_clip import create_model_from_pretrained


class ImageEncoder(nn.Module):
    def __init__(self, embed_dim=768):
        super().__init__()
        model, _ = create_model_from_pretrained(
            'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
        )
        self.backbone = model.visual.trunk
        biomed_dim = self.backbone.embed_dim  # 768 for ViT-B/16
        self.fc = nn.Linear(biomed_dim, embed_dim)
        for p in self.backbone.parameters():
            p.requires_grad = True

    def forward(self, x):
        features = self.backbone(x)  # (B, 768)
        return self.fc(features)


class LocaliserEncoder(nn.Module):
    def __init__(self, embed_dim=768):
        super().__init__()
        model, _ = create_model_from_pretrained(
            'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
        )
        self.backbone = model.visual.trunk
        biomed_dim = self.backbone.embed_dim
        self.fc = nn.Linear(biomed_dim, embed_dim)
        for p in self.backbone.parameters():
            p.requires_grad = True

    def forward(self, x):
        features = self.backbone(x)
        return self.fc(features)


class TabularMLP(nn.Module):
    def __init__(self, tab_dim, d_model, dropout=0.3):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(tab_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, d_model),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, tab):
        return self.mlp(tab)


class OCT_Pred(nn.Module):
    def __init__(self, n_labels, tab_dim, embed_dim=768, d_model=None, dropout=0.3):
        super().__init__()
        if d_model is None:
            d_model = embed_dim

        self.image_encoder = ImageEncoder(embed_dim)
        self.localiser_encoder = LocaliserEncoder(embed_dim)
        self.tab_encoder = TabularMLP(tab_dim, d_model, dropout)

        fusion_dim = 2 * embed_dim + d_model

        self.mlp = nn.Sequential(
            nn.Linear(fusion_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, n_labels),
        )

    def forward(self, image, localiser, tab):
        img_features = self.image_encoder(image)
        loc_features = self.localiser_encoder(localiser)
        tab_features = self.tab_encoder(tab)

        combined = torch.cat([img_features, loc_features, tab_features], dim=-1)
        output = self.mlp(combined)
        return output, combined


class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)

In [8]:
from sklearn.metrics import classification_report
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_predicted = []

    pbar = tqdm(loader, desc="Training", leave=False)
    for images, localisers, tab, labels in pbar:
        images = images.to(device)
        localisers = localisers.to(device)
        tab = tab.to(device)
        labels = labels.long().to(device)

        optimizer.zero_grad()
        outputs, _ = model(images, localisers, tab)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

        all_labels.extend(labels.cpu().numpy())
        all_predicted.extend(predicted.cpu().numpy())

    accuracy = 100. * correct / total
    print(classification_report(all_labels, all_predicted))
    return running_loss / len(loader), accuracy

In [9]:
from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix, cohen_kappa_score, classification_report
import numpy as np

def specificity(class_ground_truth, class_prediction):
    eps = 0.000001
    cnf_matrix = confusion_matrix(class_ground_truth, class_prediction)
    FP = cnf_matrix.sum(axis=0) - np.diag(cnf_matrix)
    FN = cnf_matrix.sum(axis=1) - np.diag(cnf_matrix)
    TP = np.diag(cnf_matrix)
    TN = cnf_matrix.sum() - (FP + FN + TP)
    FP, FN, TP, TN = FP.astype(float), FN.astype(float), TP.astype(float), TN.astype(float)
    spe = TN / (TN + FP + eps)
    return spe.mean()

def evaluate(model, dataloader, criterion, device, show_report=True):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Evaluation', leave=False)
        for images, localisers, tab, labels in pbar:
            images = images.to(device)
            localisers = localisers.to(device)
            tab = tab.to(device)
            labels = labels.long().to(device)

            outputs, _ = model(images, localisers, tab)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = 100. * correct / total
    f1 = f1_score(all_labels, all_preds, average='micro')
    rk_corr = matthews_corrcoef(all_labels, all_preds)
    spec = specificity(all_labels, all_preds)
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    score = 0.1 * f1 + 0.1 * spec + 0.6 * qwk + 0.2 * rk_corr

    if show_report:
        print("\n" + "="*80)
        print("CLASSIFICATION REPORT - Per-Class Performance:")
        print("="*80)
        print(classification_report(all_labels, all_preds,
                                    target_names=['Class 0', 'Class 1', 'Class 2'],
                                    digits=4))
        print("="*80 + "\n")

    return {
        'loss': total_loss / len(dataloader),
        'accuracy': accuracy,
        'F1_score': f1,
        'Rk-correlation': rk_corr,
        'Specificity': spec,
        'Quadratic-weighted_Kappa': qwk,
        'score': score
    }




In [10]:
model = OCT_Pred(n_labels=n_labels, tab_dim=7).to(device)
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
num_epochs = 5

best_score = -float('inf')

for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"{'='*50}")

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.2f}%")
    print(f"Val Score: {val_metrics['score']:.4f}")

    if val_metrics['score'] > best_score:
        best_score = val_metrics['score']
        best_model=model
        print(f"New best model saved! Score: {best_score:.4f}")



Epoch 1/5


              precision    recall  f1-score   support

           0       0.50      0.50      0.50      2718
           1       0.37      0.02      0.03      2720
           2       0.45      0.89      0.60      2644

    accuracy                           0.47      8082
   macro avg       0.44      0.47      0.38      8082
weighted avg       0.44      0.47      0.38      8082




CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.0565    0.3618    0.0977       351
     Class 1     0.0000    0.0000    0.0000      2821
     Class 2     0.1507    0.3646    0.2132       650

    accuracy                         0.0952      3822
   macro avg     0.0690    0.2421    0.1036      3822
weighted avg     0.0308    0.0952    0.0452      3822


Train Loss: 0.6965 | Train Acc: 46.61%
Val Loss: 2.2951 | Val Acc: 9.52%
Val Score: 0.0087
New best model saved! Score: 0.0087

Epoch 2/5


              precision    recall  f1-score   support

           0       0.55      0.71      0.62      2717
           1       0.90      0.01      0.03      2665
           2       0.55      0.93      0.69      2700

    accuracy                           0.55      8082
   macro avg       0.67      0.55      0.45      8082
weighted avg       0.67      0.55      0.45      8082




CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.0586    0.1595    0.0858       351
     Class 1     0.0000    0.0000    0.0000      2821
     Class 2     0.1744    0.7692    0.2843       650

    accuracy                         0.1455      3822
   macro avg     0.0777    0.3096    0.1234      3822
weighted avg     0.0350    0.1455    0.0562      3822


Train Loss: 0.5040 | Train Acc: 55.30%
Val Loss: 2.5437 | Val Acc: 14.55%
Val Score: 0.0683
New best model saved! Score: 0.0683

Epoch 3/5


...


CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.0704    0.4587    0.1220       351
     Class 1     1.0000    0.0028    0.0057      2821
     Class 2     0.1402    0.3292    0.1967       650

    accuracy                         0.1002      3822
   macro avg     0.4035    0.2636    0.1081      3822
weighted avg     0.7684    0.1002    0.0488      3822


Train Loss: 0.3474 | Train Acc: 62.58%
Val Loss: 2.3213 | Val Acc: 10.02%
Val Score: 0.0185

Epoch 4/5


              precision    recall  f1-score   support

           0       0.61      0.95      0.74      2663
           1       0.93      0.12      0.22      2814
           2       0.72      0.99      0.83      2605

    accuracy                           0.67      8082
   macro avg       0.75      0.69      0.60      8082
weighted avg       0.76      0.67      0.59      8082




CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.1154    0.6125    0.1942       351
     Class 1     0.6121    0.1471    0.2372      2821
     Class 2     0.1553    0.3062    0.2061       650

    accuracy                         0.2169      3822
   macro avg     0.2943    0.3553    0.2125      3822
weighted avg     0.4888    0.2169    0.2280      3822


Train Loss: 0.2286 | Train Acc: 67.33%
Val Loss: 1.8657 | Val Acc: 21.69%
Val Score: 0.1179
New best model saved! Score: 0.1179

Epoch 5/5


              precision    recall  f1-score   support

           0       0.66      0.97      0.79      2734
           1       0.96      0.30      0.45      2671
           2       0.82      1.00      0.90      2677

    accuracy                           0.76      8082
   macro avg       0.82      0.76      0.71      8082
weighted avg       0.81      0.76      0.72      8082




CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.1265    0.6467    0.2116       351
     Class 1     0.6972    0.3215    0.4401      2821
     Class 2     0.0744    0.0831    0.0785       650

    accuracy                         0.3108      3822
   macro avg     0.2993    0.3504    0.2434      3822
weighted avg     0.5388    0.3108    0.3576      3822


Train Loss: 0.1531 | Train Acc: 75.77%
Val Loss: 1.7526 | Val Acc: 31.08%
Val Score: 0.1022


In [11]:
results = evaluate(best_model, test_loader, criterion, device)
print(f"Accuracy: {results['accuracy']:.2f}%")
print(f"F1 Score: {results['F1_score']:.4f}")
print(f"Quadratic Weighted Kappa: {results['Quadratic-weighted_Kappa']:.4f}")
print(f"Rk Correlation (Spearman): {results['Rk-correlation']:.4f}")
print(f"Specificity: {results['Specificity']:.4f}")
print(f"Score: {results['score']:.4f}")


CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.0957    0.5208    0.1617       578
     Class 1     0.8452    0.2349    0.3677      4810
     Class 2     0.0350    0.1340    0.0555       321

    accuracy                         0.2582      5709
   macro avg     0.3253    0.2965    0.1950      5709
weighted avg     0.7237    0.2582    0.3293      5709


Accuracy: 25.82%
F1 Score: 0.2582
Quadratic Weighted Kappa: -0.0354
Rk Correlation (Spearman): -0.0172
Specificity: 0.6652
Score: 0.0677


In [12]:
from google.colab import drive
drive.mount('/content/drive')
#from google.colab import files
#files.download('model.pth')
import torch
torch.save(best_model.state_dict(), '/content/drive/MyDrive/best_model_mlp2.pth')

ValueError: mount failed